In [1]:
import cProfile
import numpy as np
import pandas as pd
from src.archive.recovery_model import RecoveryModel

pd.set_option("multi_sparse", False)

In [2]:
# Select a folder for the data to be used
folder = "test_2"  # test_1  test_2  Toy_WEEE_v2
input_format = "csv"
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
layer_4 = "element"

In [3]:
layer_names = (layer_0, layer_1, layer_2, layer_3, layer_4)

composition_dct = {
    "url": f"data/{folder}/composition.{input_format}",
    "sheet": "composition",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Layer 4": layer_4,
        "Value": "data",
        "Year": "year",
        "parameterCode": "layer_code",
        "Scenario": "scenario",
        "Location": "region",
        "UoM": "unit",
    },
    "data_processing": {
        layer_2: ("c-p",),
        layer_3: ("m-p", "m-c"),
        layer_4: ("e-p", "e-c", "e-m"),
    },
}

tc_dct = {
    "url": f"data/{folder}/TCs.{input_format}",
    "sheet": "TCs",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
    },
    "all_symbol": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        layer_4: "E*",
    },
}

inputs_dct = {
    "url": f"data/{folder}/inputs.{input_format}",
    "sheet": "inputs",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Year": "year",
    },
}

---

# Recovery model


In [4]:
weee = RecoveryModel(
    composition_dct=composition_dct,
    inputs_dct=inputs_dct,
    tc_dct=tc_dct,
    layer_names=layer_names,
    save_intermediary_steps=False,
    input_format=input_format,
)

---

# Metadata


In [5]:
print(weee)

{   'component': ('C1', 'C2', 'C3'),
    'element': ('E1', 'E2'),
    'flow': ('F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8'),
    'index': {   'component': {'C1': 1, 'C2': 2, 'C3': 3, '∅': 0},
                 'element': {'E1': 1, 'E2': 2, '∅': 0},
                 'flow': {   'F1': 0,
                             'F2': 1,
                             'F3': 2,
                             'F4': 3,
                             'F5': 4,
                             'F6': 5,
                             'F7': 6,
                             'F8': 7},
                 'material': {'M1': 1, 'M2': 2, '∅': 0},
                 'product': {'P1': 1, 'P2': 2, '∅': 0}},
    'material': ('M1', 'M2'),
    'product': ('P1', 'P2')}


In [6]:
weee.var

{'flow': ('F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8'),
 'product': ('P1', 'P2'),
 'component': ('C1', 'C2', 'C3'),
 'material': ('M1', 'M2'),
 'element': ('E1', 'E2'),
 'index': {'flow': {'F1': 0,
   'F2': 1,
   'F3': 2,
   'F4': 3,
   'F5': 4,
   'F6': 5,
   'F7': 6,
   'F8': 7},
  'product': {'∅': 0, 'P1': 1, 'P2': 2},
  'component': {'∅': 0, 'C1': 1, 'C2': 2, 'C3': 3},
  'material': {'∅': 0, 'M1': 1, 'M2': 2},
  'element': {'∅': 0, 'E1': 1, 'E2': 2}}}

In [7]:
weee.index

MultiIndex([('F1',  '∅',  '∅',  '∅',  '∅'),
            ('F1',  '∅',  '∅',  '∅', 'E1'),
            ('F1',  '∅',  '∅',  '∅', 'E2'),
            ('F1',  '∅',  '∅', 'M1',  '∅'),
            ('F1',  '∅',  '∅', 'M1', 'E1'),
            ('F1',  '∅',  '∅', 'M1', 'E2'),
            ('F1',  '∅',  '∅', 'M2',  '∅'),
            ('F1',  '∅',  '∅', 'M2', 'E1'),
            ('F1',  '∅',  '∅', 'M2', 'E2'),
            ('F1',  '∅', 'C1',  '∅',  '∅'),
            ...
            ('F8', 'P2', 'C2', 'M2', 'E2'),
            ('F8', 'P2', 'C3',  '∅',  '∅'),
            ('F8', 'P2', 'C3',  '∅', 'E1'),
            ('F8', 'P2', 'C3',  '∅', 'E2'),
            ('F8', 'P2', 'C3', 'M1',  '∅'),
            ('F8', 'P2', 'C3', 'M1', 'E1'),
            ('F8', 'P2', 'C3', 'M1', 'E2'),
            ('F8', 'P2', 'C3', 'M2',  '∅'),
            ('F8', 'P2', 'C3', 'M2', 'E1'),
            ('F8', 'P2', 'C3', 'M2', 'E2')],
           names=['flow', 'product', 'component', 'material', 'element'], length=864)

---

# Linear equations


In [8]:
weee.lneqs

<864x864 sparse matrix of type '<class 'numpy.float64'>'
	with 441 stored elements in Compressed Sparse Row format>

In [9]:
rows_with_TCs = np.argwhere(weee.lneqs.getnnz(axis=1)).squeeze()
print(rows_with_TCs.size)
rows_with_TCs

318


array([ 45,  48,  49,  50,  51,  52,  53,  54,  57,  58,  59,  60,  61,
        62,  63,  66,  67,  68,  69,  70,  71,  81,  84,  85,  86,  87,
        88,  89,  90,  93,  94,  95,  96,  97,  98,  99, 102, 103, 104,
       105, 106, 107, 126, 127, 128, 129, 130, 131, 132, 133, 134, 153,
       154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166,
       167, 168, 169, 170, 189, 190, 191, 192, 193, 194, 195, 196, 197,
       198, 199, 200, 201, 202, 203, 204, 205, 206, 234, 235, 236, 237,
       238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250,
       251, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281,
       282, 283, 284, 285, 286, 287, 306, 307, 308, 309, 310, 311, 312,
       313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 342, 343,
       344, 345, 346, 347, 348, 349, 350, 378, 379, 380, 381, 382, 383,
       384, 385, 386, 414, 415, 416, 417, 418, 419, 420, 421, 422, 444,
       445, 446, 447, 448, 449, 453, 454, 455, 456, 457, 458, 48

In [10]:
i = rows_with_TCs[10]
sparr = weee.lneqs.getrow(i)
rows, cols = sparr.nonzero()
pd.DataFrame(
    data=sparr.data,
    index=weee.index[cols],
    columns=[weee.index[i]],
).transpose()  # transpose() to get same structure as I/O tables

flow,F1
product,P1
component,C2
material,M1
element,∅
"(F1, P1, C2, M1, E2)",0.72


In [11]:
start = 0
end = 10

spmtx = weee.lneqs[rows_with_TCs[start:end], :]
if spmtx.data.size:
    s = pd.Series.sparse.from_coo(spmtx.tocoo(), dense_index=False)
    s = s.unstack().rename(index=dict(enumerate(rows_with_TCs[start:end])))
    s.index = weee.index[s.index]
    s.columns = weee.index[s.columns]
    # transpose to get same structure as I/O tables
    display(s.transpose().astype(str).replace("nan", ""))
else:
    print("no TCs")

,,,,flow,F1,F1,F1,F1,F1,F1,F1,F1,F1,F1
,,,,product,P1,P1,P1,P1,P1,P1,P1,P1,P1,P1
,,,,component,C1,C1,C1,C1,C1,C1,C1,C2,C2,C2
,,,,material,∅,M1,M1,M1,M2,M2,M2,∅,M1,M1
,,,,element,∅,∅,E1,E2,∅,E1,E2,∅,∅,E1
flow,product,component,material,element,,,,,,,,,,
F1,P1,∅,∅,∅,0.25,,,,,,,0.59,,
F1,P1,C1,∅,∅,,0.52,,,0.48,,,,,
F1,P1,C1,M1,∅,,,0.7,0.3,,,,,,
F1,P1,C1,M2,∅,,,,,,0.98,0.02,,,
F1,P1,C2,∅,∅,,,,,,,,,0.66,


In [12]:
idx = pd.IndexSlice["F2", "P1", "C1", :, :]
# ! This only works for test_1 and test_2


idxs = weee.get_indexer(idx)
spmtx = weee.lneqs[idxs, :]
if spmtx.data.size:
    s = pd.Series.sparse.from_coo(spmtx.tocoo(), dense_index=False)
    s = s.unstack().rename(index=dict(enumerate(idxs)))
    s.index = weee.index[s.index]
    s.columns = weee.index[s.columns]
    # transpose to get same structure as I/O tables
    display(s.transpose().astype(str).replace("nan", ""))
else:
    print("no TCs")

,,,,flow,F2,F2,F2,F2,F2,F2,F2,F2,F2
,,,,product,P1,P1,P1,P1,P1,P1,P1,P1,P1
,,,,component,C1,C1,C1,C1,C1,C1,C1,C1,C1
,,,,material,∅,∅,∅,M1,M1,M1,M2,M2,M2
,,,,element,∅,E1,E2,∅,E1,E2,∅,E1,E2
flow,product,component,material,element,,,,,,,,,
F1,P1,C1,∅,∅,0.25,,,,,,,,
F1,P1,C1,∅,E1,,0.25,,,,,,,
F1,P1,C1,∅,E2,,,0.25,,,,,,
F1,P1,C1,M1,∅,,,,0.25,,,,,
F1,P1,C1,M1,E1,,,,,0.25,,,,


In [13]:
idx = pd.IndexSlice["F2", "∅", "C1", :, :]
# ! This only works for test_1 and test_2

idxs = weee.get_indexer(idx)
spmtx = weee.lneqs[idxs, :]
if spmtx.data.size:
    s = pd.Series.sparse.from_coo(spmtx.tocoo(), dense_index=False)
    s = s.unstack().rename(index=dict(enumerate(idxs)))
    s.index = weee.index[s.index]
    s.columns = weee.index[s.columns]
    # transpose to get same structure as I/O tables
    display(s.transpose().astype(str).replace("nan", ""))
else:
    print("no TCs")

no TCs


In [14]:
idx = pd.IndexSlice["F5", :, "C1", :, "E1"]
# ! This only works for test_1 and test_2

idxs = weee.get_indexer(idx)
spmtx = weee.lneqs[idxs, :]
if spmtx.data.size:
    s = pd.Series.sparse.from_coo(spmtx.tocoo(), dense_index=False)
    s = s.unstack().rename(index=dict(enumerate(idxs)))
    s.index = weee.index[s.index]
    s.columns = weee.index[s.columns]
    # transpose to get same structure as I/O tables
    display(s.transpose().astype(str).replace("nan", ""))
else:
    print("no TCs")

,,,,flow,F5,F5,F5,F5,F5,F5
,,,,product,∅,∅,P1,P1,P2,P2
,,,,component,C1,C1,C1,C1,C1,C1
,,,,material,M1,M2,M1,M2,M1,M2
,,,,element,E1,E1,E1,E1,E1,E1
flow,product,component,material,element,,,,,,
F2,∅,C1,M1,E1,0.39,,,,,
F2,∅,C1,M2,E1,,0.32,,,,
F2,P1,C1,M1,E1,,,0.39,,,
F2,P1,C1,M2,E1,,,,0.32,,
F2,P2,C1,M1,E1,,,,,0.39,


In [15]:
mask = weee.index.get_level_values(-1) != "∅"
idxs = weee.get_indexer(weee.index[mask])
spmtx = weee.lneqs.T[idxs, :]
if spmtx.data.size:
    s = pd.Series.sparse.from_coo(spmtx.tocoo(), dense_index=False)
    s = s.unstack().rename(index=dict(enumerate(idxs)))
    s.index = weee.index[s.index]
    s.columns = weee.index[s.columns]
    # transpose to get same structure as I/O tables
    # display(s.transpose().astype(str).replace("nan", ""))
    display(s.head(30).astype(str).replace("nan", ""))
else:
    print("no TCs")

,,,,flow,F2,F2,F2,F2,F2,F2,F2,F2,F2,F2,...,F8,F8,F8,F8,F8,F8,F8,F8,F8,F8
,,,,product,∅,∅,∅,∅,∅,∅,P1,P1,P1,P1,...,P1,P1,P2,P2,P2,P2,P2,P2,P2,P2
,,,,component,C2,C2,C2,C2,C2,C2,C1,C1,C1,C1,...,C3,C3,∅,∅,C1,C1,C2,C2,C3,C3
,,,,material,∅,∅,M1,M1,M2,M2,∅,∅,M1,M1,...,M1,M2,M1,M2,M1,M2,M1,M2,M1,M2
,,,,element,E1,E2,E1,E2,E1,E2,E1,E2,E1,E2,...,E1,E2,E1,E2,E1,E2,E1,E2,E1,E2
flow,product,component,material,element,,,,,,,,,,,,,,,,,,,,,
F1,P1,C1,∅,E1,,,,,,,0.25,,,,...,,,,,,,,,,
F1,P1,C1,∅,E2,,,,,,,,0.25,,,...,,,,,,,,,,
F1,P1,C1,M1,E1,,,,,,,,,0.25,,...,,,,,,,,,,
F1,P1,C1,M1,E2,,,,,,,,,,0.25,...,,,,,,,,,,
F1,P1,C1,M2,E1,,,,,,,,,,,...,,,,,,,,,,


In [16]:
P = s.index.get_level_values(1)
C = s.index.get_level_values(2)
M = s.index.get_level_values(3)
mask1 = (P != "∅") & (C == "∅")
mask2 = (C != "∅") & (M == "∅")
idx = s.loc[~(mask1 | mask2)].index.values

P = s.columns.get_level_values(1)
C = s.columns.get_level_values(2)
M = s.columns.get_level_values(3)
mask1 = (P != "∅") & (C == "∅")
mask2 = (C != "∅") & (M == "∅")
cols = s.loc[:, ~(mask1 | mask2)].columns.values

s.loc[idx, cols].to_csv("test_elements_only.csv")
display(s.loc[idx, cols].head(30).astype(str).replace("nan", ""))

,,,,flow,F2,F2,F2,F2,F2,F2,F2,F2,F2,F2,...,F8,F8,F8,F8,F8,F8,F8,F8,F8,F8
,,,,product,∅,∅,∅,∅,P1,P1,P1,P1,P1,P1,...,P1,P1,P1,P1,P2,P2,P2,P2,P2,P2
,,,,component,C2,C2,C2,C2,C1,C1,C1,C1,C2,C2,...,C2,C2,C3,C3,C1,C1,C2,C2,C3,C3
,,,,material,M1,M1,M2,M2,M1,M1,M2,M2,M1,M1,...,M1,M2,M1,M2,M1,M2,M1,M2,M1,M2
,,,,element,E1,E2,E1,E2,E1,E2,E1,E2,E1,E2,...,E1,E2,E1,E2,E1,E2,E1,E2,E1,E2
flow,product,component,material,element,,,,,,,,,,,,,,,,,,,,,
F1,P1,C1,M1,E1,,,,,0.25,,,,,,...,,,,,,,,,,
F1,P1,C1,M1,E2,,,,,,0.25,,,,,...,,,,,,,,,,
F1,P1,C1,M2,E1,,,,,,,0.25,,,,...,,,,,,,,,,
F1,P1,C1,M2,E2,,,,,,,,0.25,,,...,,,,,,,,,,
F1,P1,C2,M1,E1,,,,,,,,,0.04,,...,,,,,,,,,,


MASS BALANCE


In [17]:
df = pd.read_csv(tc_dct["url"]).rename(columns=tc_dct["mapper"])[["inflows", "outflows", "process"]].drop_duplicates()
df

,inflows,outflows,process
0,F1,F2,T1
4,F1,F3,T1
8,F6,F2,T1
9,F6,F3,T1
11,F2,F4,T2
12,F2,F5,T2
16,F3,F6,T4
18,F4,F6,T4
19,F3,F7,T4
23,F4,F7,T4


In [18]:
pd.concat([df.groupby("process")["inflows"].unique(), df.groupby("process")["outflows"].unique()], axis=1)

,inflows,outflows
process,,
T1,"[F1, F6]","[F2, F3]"
T2,[F2],"[F4, F5]"
T3,"[F5, F7]",[F8]
T4,"[F3, F4]","[F6, F7]"


In [20]:
arr1 = (pd.concat([pd.get_dummies(df["inflows"]), df["process"]], axis=1).groupby("process").any()).astype(int)
arr2 = -(pd.concat([pd.get_dummies(df["outflows"]), df["process"]], axis=1).groupby("process").any()).astype(int)

arr1, arr2 = arr1.align(arr2, fill_value=0)
arr1.add(arr2).T

process,T1,T2,T3,T4
F1,1,0,0,0
F2,-1,1,0,0
F3,-1,0,0,1
F4,0,-1,0,1
F5,0,-1,1,0
F6,1,0,0,-1
F7,0,0,1,-1
F8,0,0,-1,0


---

# Constant terms


In [15]:
weee.y

<864x1 sparse array of type '<class 'numpy.int64'>'
	with 2 stored elements in Compressed Sparse Column format>

In [16]:
rows, _ = weee.y.nonzero()
pd.Series(
    data=weee.y.data,
    index=weee.index[rows],
)

flow  product  component  material  element
F1    P1       ∅          ∅         ∅          1000
F1    P2       ∅          ∅         ∅           700
dtype: int64

---

# Solver


In [17]:
solution = weee.solve(expand=False)
solution

flow  product  component  material  element
F1    P1       ∅          ∅         ∅          1000.000000
F1    P1       C1         ∅         ∅           250.000000
F1    P1       C1         M1        ∅           130.000000
F1    P1       C1         M1        E1           91.000000
F1    P1       C1         M1        E2           39.000000
                                                  ...     
F8    P1       C3         M2        E2            0.404007
F8    P2       C1         M1        E1            0.228544
F8    P2       C1         M2        E2            0.086549
F8    P2       C2         M1        E1            0.282068
F8    P2       C2         M2        E2            0.073321
Name: mass, Length: 180, dtype: float64

In [18]:
weee.solve(expand=True)

flow  product  component  material  element
F1    P1       C1         M1        E1          91.000000
F1    P2       C1         M1        E1          40.695200
F1    P1       C1         M1        E2          39.000000
F1    P2       C1         M1        E2         115.824800
F1    P1       C1         M1        ∅          130.000000
                                                  ...    
F8    P1       C3         M2        ∅            0.404007
F8    P1       C3         ∅         ∅            0.441213
F8    P1       ∅          ∅         ∅            2.840704
F8    P2       ∅          ∅         ∅            0.670482
F8    ∅        ∅          ∅         ∅            3.511187
Name: mass, Length: 224, dtype: float64

---

# Performance


In [ ]:
cProfile.run(
    "RecoveryModel(composition_dct, inputs_dct, tc_dct ,layer_names, input_format, False)",
    "results/performances/RecoveryModel.pstats",
)

cProfile.run(
    "weee.solve(expand=False)",
    "results/performances/Solver.pstats",
)

# # ! then go into results/performances/ and run the following commands
# # (-n 2 means cutoff at 2%)
# gprof2dot -f pstats -n 2 RecoveryModel.pstats | dot -Tpng -o RecoveryModel.png
# gprof2dot -f pstats -n 2 Solver.pstats | dot -Tpng -o Solver.png

In [ ]:
cProfile.run(
    "RecoveryModel(composition_dct, inputs_dct, tc_dct ,layer_names, input_format, False)",
    "results/performances/RecoveryModel.prof",
)

# cProfile.run(
#     "weee.solve(expand=False)",
#     "results/performances/Solver.prof",
# )

# # ! then go into results/performances/ and run the following commands
# snakeviz RecoveryModel.prof
# snakeviz Solver.prof